In [1]:
!pip install langchain_community langchain_google_genai langchain-chroma>=0.1.2 python-docx

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.9.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-exporter-otlp-proto-common==1.37.0, but you have opentelemetry-exporter-otlp-proto-common 1.38.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-proto==1.37.0, but you have opentelemetry-proto 1.38.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-sdk~=1.37.0, but you have opentelemetry-sdk 1.38.0 which is incompatible.
google-adk 1.19.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.38.0 whi

In [5]:
import os
import getpass
from IPython.display import Markdown, display

from docx import Document as wd

#Langchain mensajes
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage

#Langchain prompts
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompt_values import ChatPromptValue

#Langchain chat model (Google)
from langchain_google_genai import ChatGoogleGenerativeAI

#Langchain cadena
from langchain_classic.chains import RetrievalQA
from langchain_core.runnables.base import RunnableSequence

#Langchain document loader
from langchain_core.documents.base import Document
from langchain_community.document_loaders import WebBaseLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

#Langchain embedding model (Google)
from langchain_google_genai import GoogleGenerativeAIEmbeddings

#Langchain vector store (Chroma)
from langchain_chroma import Chroma
from langchain_core.vectorstores.base import VectorStoreRetriever

In [6]:
# Ingresa el API key
os.environ["GOOGLE_API_KEY"]: str = getpass.getpass("Ingresa tu Google AI API key: ")

Ingresa tu Google AI API key: ··········


### **Generador de documentos**

El objetivo de este notebook es generar un documento con las secciones especificadas por el usuario. El documento generada será en base a la información de un artículo web.

#### Cargar documento desde la web

In [7]:
#WebBaseLoader carga información de la web y transforma el código html en documentos - requiere la dirección de la página web
loader: WebBaseLoader = WebBaseLoader('https://www.infobae.com/peru/2025/05/03/machu-picchu-es-clave-estos-son-los-5-destinos-turisticos-mas-visitados-del-peru-en-lo-que-va-del-2025/')

#Luego de instanciar la clase, se pueden cargar los documentos con el método load
documento_web: list = loader.load()

#### Dividir el documento en chunks

In [8]:
# Este loader trata de separa el texto primero en párrafos, luego en oraciones y por último en palábras
splitter_documento:RecursiveCharacterTextSplitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=20,
    length_function=len
)

#Con el método split_documentos se dividen los documentos
textos_split_recursive: list[Document] = splitter_documento.split_documents(documento_web)

#### Iniciar nuestro modelo chat y el de embeddings

In [9]:
#El módulo GoogleGenerativeAIEmbeddings nos permite acceder al modelo de gemini
embedding_model: GoogleGenerativeAIEmbeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

#Inicializamos nuestro modelo chat
model: ChatGoogleGenerativeAI = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

#### Iniciar e indexar nuestra información en una base vectorizada

In [10]:
# Se puede inicializar la base de datos de la siguiente manera, si se quiere persistir los datos de chroma se debe utilizar el argumento persist_directory
vector_store: Chroma = Chroma(collection_name="Apps_IA", embedding_function=embedding_model, persist_directory="./chroma_db")

#Para indexar los documentos en la base de datos vectorizada se debe usar el método from_documents
db_RAG: Chroma = vector_store.from_documents(textos_split_recursive, embedding_model)

#Esta sentencia nos servirá para utilizar nuestra base de datos vectorizada para recuperar secciones de texto relevantes
retriever: VectorStoreRetriever = db_RAG.as_retriever(search_kwargs={"k": 3})

#### Crear las instrucciones para nuestro sistema RAG

In [11]:
#Creamos las instrucciones para nuestro generador de documentos
mensaje_sistema: str = """
Eres un asistente experto en generar documentos claros y concisos a partir de documentos recuperados mediante RAG.
Tu tarea es producir texto listo para insertar en Word, manteniendo coherencia y claridad.
Debes incluir información relevante, cifras, subtítulos y listas si aplica, y señalar explícitamente si falta información.
Evita Markdown, hashtags (#) o cualquier formato de código.
"""

mensaje_humano: str = """
Genera un texto para la sección: {question}

Contexto: {context}

Sigue estas reglas y estructura:

1. Texto principal: descripción de la sección y contexto.
2. Subtítulos (si aplica): en negrita o como párrafos separados.
3. Listas de datos clave: viñetas o numeración.
4. Advertencia si falta información: "No se dispone de información adicional para esta sección."

Reglas adicionales:
- Mantén coherencia en cifras y nombres propios.
- Mantén un estilo formal, legible y conciso.
- Todo el texto debe estar listo para Word (sin Markdown ni #).

Ejemplos:

Ejemplo 1 – Turismo:
Sección: Destinos más visitados
Machu Picchu se posiciona como el destino turístico más visitado del Perú en 2025, con 191,351 visitantes en el primer bimestre, un aumento de 16.7% respecto al año anterior. La composición de visitantes es 63% extranjeros y 37% nacionales.
No se dispone de información detallada sobre los otros cuatro destinos más visitados.

Ejemplo 2 – Ciencia:
Sección: Avances en Inteligencia Artificial
En 2025, se reportó un incremento del 23% en publicaciones científicas sobre modelos de lenguaje grande. Se destacan mejoras en eficiencia de entrenamiento y reducción de sesgos.
No se dispone de información sobre publicaciones en países específicos.

Evalúa la calidad según estos criterios:
- Coherencia, claridad, formato listo para Word, completitud, estilo.
- Si faltan datos, incluye la advertencia correspondiente.
"""

In [12]:
#Creamos nuestra plantilla prompt: Es importante definir una variable donde colocaremos nuestro contexto, el cual será colocado dinámicamente
prompt: ChatPromptValue = ChatPromptTemplate( [SystemMessage(content=mensaje_sistema), ("human", mensaje_humano)])

#### Crear nuestra cadena RAG

Documentación RetrievalQA: https://python.langchain.com/api_reference/langchain/chains/langchain.chains.retrieval_qa.base.RetrievalQA.html

In [13]:
#Creamos nuestra cadena RAG con el módulo RetrievalQA
qa_chain: RetrievalQA = RetrievalQA.from_chain_type(
    llm=model,
    retriever=retriever,
    return_source_documents=False,
    chain_type="stuff",
    chain_type_kwargs={
        "prompt": prompt
    }
)

#### Generar nuestro documento

In [14]:
#Lista de secciones

secciones = [
    "Destinos más visitados",
    "Estadísticas de visitantes",
    "Importancia de Machu Picchu",
]

#Creación de documento Word
doc_word = wd()
doc_word.add_heading("Documento automatizado IA", 0)

for seccion in secciones:
    doc_word.add_heading(seccion, level=1)

    #Ejecutamos nuestra cadena RAG
    resultado = qa_chain.invoke({"query": seccion})

    contenido = resultado["result"]

    #Añadimos el documento
    doc_word.add_paragraph(contenido)

# Guardamos documento
doc_word.save("Guia_RAG_con_fuentes.docx")
print("Documento generado con fuentes en Guia_RAG_con_fuentes.docx")

Documento generado con fuentes en Guia_RAG_con_fuentes.docx
